# 14 -- Cow: milk lifecycle

Second animal after goose (`13_goose.ipynb`). Decisions from chat, before
writing any code:

- **Single species first, COW** (not both COW/SHEEP at once): reading
  `kaggriculture.py` directly showed `_daily_refresh_animals` and the
  `FEED`/`COLLECT_FERTILIZER`/`CARE`/`HARVEST` action handlers are the
  *same* function for `COOP` and `PASTURE` -- only `interval`/
  `first_yield_day`/`max_held` differ in the `ANIMALS` table, and the
  agent's own logic never reads those directly (unlike crop harvest
  thresholds, which are hardcoded per crop). So the only genuinely new,
  unproven thing is whether `PASTURE` behaves the same way as `COOP` --
  worth isolating with one species before the second (cheap once the
  first works).
- **Skip the isolated mini-test stage** goose went through (its
  `_test_goose_agent`) -- the engine-source read above already de-risks
  most of the mechanism. Going straight to a hand inside
  `melon_then_wheat_goose_agent`.
- **Hand 2 gets repurposed for the cow**, same as hand 4 was for the
  goose -- not a new hire. `HAND_ZONES[2]` (`(4,0)-(4,1)`, 2 tiles) is the
  zone being sacrificed for the `PASTURE` tile; no other NW tile is free
  (all 13 are already claimed by `HAND_ZONES[0..4]`/`GOOSE_TILE`).
- Open decisions, to work out below: which of `(4,0)`/`(4,1)` becomes
  `PASTURE_TILE`, and hand 2's own `BUILD_PASTURE`/`PLACE`/
  `FEED`/`COLLECT_FERTILIZER`/`CARE`/`HARVEST` logic (mirror hand 4's
  block in `melon_then_wheat_goose_agent`, adapted for `COW`/`MILK`/
  `PASTURE`).

In [1]:
import sys
from pathlib import Path

from kaggle_environments import make

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kaggriculture_bot.evaluate import evaluate
from kaggriculture_bot.strategies.baseline import (
    DIRECTIONS,
    GOOSE_FEEDER,
    GOOSE_TILE,
    HAND_ZONES,
    LOOP_ORDER_12,
    LOOP_ORDER_HANDS,
    SHED_ADJACENT,
    SPAWN_FEEDER,
    melon_then_wheat_goose_agent,
)

## Decide `PASTURE_TILE`

Which of `(4,0)`/`(4,1)` is closer to the shed (`(4,4)`/`(5,4)`/`(4,5)`/
`(5,5)`)? Same "closest to shed" heuristic used to place `GOOSE_TILE` at
`(4,2)` (the closest zoned tile to the shed within hand 4's own zone).

In [2]:
# TODO: pick (4,0) or (4,1) -- whichever is closer to the shed.
PASTURE_TILE = (4,1)

# TODO: a direct route from the shared entry point to PASTURE_TILE, same
# role as GOOSE_FEEDER. Goose needed this (see PROGRESS.md, 2026-08-27)
# to avoid running out of turn budget partway through the shared
# LOOP_ORDER_HANDS walk -- confirm whether hand 2 hits the same problem
# before assuming it needs the same fix.
PASTURE_FEEDER = {(4, 4): (4, 3), (4, 3): (4, 2), (4, 2): (4, 1), (4, 1): (4, 1)}

## Skeleton: `melon_then_wheat_goose_cow_agent`

Copied from `melon_then_wheat_goose_agent` unchanged except hand 2's own
branch (marked `TODO` below) -- everything else (farmer, hands 0/1/3,
hand 4's goose logic, hire, seed buying, `WHEAT`/`MELON`/`FERTILIZER`
selling) stays exactly as already validated. Fill in hand 2's
`BUILD_PASTURE`/`PLACE`/`FEED`/`COLLECT_FERTILIZER`/`CARE`/`HARVEST`
branch yourself -- hand 4's block right above it is the template to
adapt (`COW`/`MILK`/`PASTURE` in place of `GOOSE`/`EGG`/`COOP`).

In [3]:
def melon_then_wheat_goose_cow_agent(obs):
    crop_to_plant = "MELON" if obs["day"] < 10 else "WHEAT"
    crop_seeds = obs["private"]["seeds"].get(crop_to_plant, 0)

    market = []
    hands = []

    if obs["step"] == 0:
        market.append(["BUY_ANIMAL", "GOOSE", 1])
        # TODO: BUY_ANIMAL COW too -- same step, or does it depend on
        # hand 2 being hired/positioned first?
        market.append(["BUY_ANIMAL","COW",1])
    if obs["day"] < 10 and obs["private"]["shed"].get("WHEAT", 0) == 0:
        market.append(["BUY_PRODUCT", "WHEAT", 1])

    for i, (hx, hy) in enumerate(obs["farms"][obs["player"]]["hands"]):
        if i == 4:
            hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **GOOSE_FEEDER}
        elif i == 2:
            hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **PASTURE_FEEDER}
        else:
            hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **HAND_ZONES[i % len(HAND_ZONES)]}
        target_x, target_y = hand_loop[(hx, hy)]
        if (target_x, target_y) == (hx, hy):
            move = ["PASS"]
        else:
            move = [DIRECTIONS[(target_x - hx, target_y - hy)]]

        hand_tile = obs["farms"][obs["player"]]["tiles"][hy][hx]

        if i == 4:
            hand_inventory = obs["private"]["inventories"][i + 1]

            if (hx, hy) == GOOSE_TILE:
                # At the coop.
                if hand_tile is None:
                    op = ["BUILD_COOP"]
                elif hand_tile.get("kind") == "COOP" and hand_tile.get("animal") != "GOOSE":
                    if hand_inventory.get("GOOSE", 0) > 0:
                        op = ["PLACE", "GOOSE", 1]
                    else:
                        op = ["PASS"]
                elif hand_inventory.get("WHEAT", 0) > 0 and not hand_tile.get("fed_today"):
                    op = ["FEED"]
                elif hand_tile.get("fertilizer_available"):
                    op = ["COLLECT_FERTILIZER"]
                elif not hand_tile.get("cared_today"):
                    op = ["CARE"]
                elif hand_tile.get("yield_units", 0) > 0:
                    op = ["HARVEST"]
                else:
                    op = ["PASS"]
                    egg_in_shed = obs["private"]["shed"].get("EGG", 0)
                    if egg_in_shed > 0:
                        market.append(["SELL", "EGG", egg_in_shed])
            else:
                # Not at the coop yet: pickup priority (GOOSE, then
                # WHEAT), only possible while shed-adjacent; otherwise
                # tend whatever tile it's standing on, same as hands 0-3.
                if hand_tile == "LOCKED":
                    op = move
                elif (hx, hy) in SHED_ADJACENT and hand_inventory.get("GOOSE", 0) == 0 and obs["private"]["shed"].get("GOOSE", 0) > 0:
                    op = ["PICKUP", "GOOSE", 1]
                elif (hx, hy) in SHED_ADJACENT and hand_inventory.get("WHEAT", 0) == 0 and obs["private"]["shed"].get("WHEAT", 0) > 0:
                    op = ["PICKUP", "WHEAT", 1]
                elif hand_tile is None and crop_seeds > 0:
                    op = ["PLANT", crop_to_plant]
                elif hand_tile is not None and hand_tile.get("kind") == "PLANT":
                    crop_age = obs["day"] - hand_tile["planted_day"]
                    threshold = 10 if hand_tile["crop"] == "MELON" else 2
                    if crop_age >= threshold:
                        op = ["HARVEST"]
                    elif not hand_tile.get("watered_today"):
                        op = ["WATER"]
                    else:
                        op = move
                else:
                    op = move
        elif i == 2:
            hand_inventory = obs["private"]["inventories"][i + 1]
            # TODO: cow/pasture logic goes here. Mirror the "if i == 4"
            # block above -- BUILD_PASTURE on an empty PASTURE_TILE,
            # PICKUP COW (then WHEAT) while shed-adjacent, PLACE COW on
            # an empty PASTURE tile, then FEED > COLLECT_FERTILIZER >
            # CARE > HARVEST > SELL MILK, same priority order as goose.
            if (hx, hy) == PASTURE_TILE:
                # At the pasture.
                if hand_tile is None:
                    op = ["BUILD_PASTURE"]
                elif hand_tile.get("kind") == "PASTURE" and hand_tile.get("animal") != "COW":
                    if hand_inventory.get("COW", 0) > 0:
                        op = ["PLACE", "COW", 1]
                    else:
                        op = ["PASS"]
                elif hand_inventory.get("WHEAT", 0) > 0 and not hand_tile.get("fed_today"):
                    op = ["FEED"]
                elif hand_tile.get("fertilizer_available"):
                    op = ["COLLECT_FERTILIZER"]
                elif not hand_tile.get("cared_today"):
                    op = ["CARE"]
                elif hand_tile.get("yield_units", 0) > 0:
                    op = ["HARVEST"]
                else:
                    op = ["PASS"]
            else:
                if hand_tile == "LOCKED":
                    op = move
                elif (hx, hy) in SHED_ADJACENT and hand_inventory.get("COW", 0) == 0 and obs["private"]["shed"].get("COW", 0) > 0:
                    op = ["PICKUP", "COW", 1]
                elif (hx, hy) in SHED_ADJACENT and hand_inventory.get("WHEAT", 0) == 0 and obs["private"]["shed"].get("WHEAT", 0) > 0:
                    op = ["PICKUP", "WHEAT", 1]
                elif hand_tile is None and crop_seeds > 0:
                    op = ["PLANT", crop_to_plant]
                elif hand_tile is not None and hand_tile.get("kind") == "PLANT":
                    crop_age = obs["day"] - hand_tile["planted_day"]
                    threshold = 10 if hand_tile["crop"] == "MELON" else 2
                    if crop_age >= threshold:
                        op = ["HARVEST"]
                    elif not hand_tile.get("watered_today"):
                        op = ["WATER"]
                    else:
                        op = move
                else:
                    op = move
  
        elif hand_tile == "LOCKED" or (hx, hy) not in HAND_ZONES[i % len(HAND_ZONES)]:
            op = move
        elif hand_tile is None and crop_seeds > 0:
            op = ["PLANT", crop_to_plant]
        elif hand_tile is not None and hand_tile.get("kind") == "PLANT":
            crop_age = obs["day"] - hand_tile["planted_day"]
            threshold = 10 if hand_tile["crop"] == "MELON" else 2
            if crop_age >= threshold:
                op = ["HARVEST"]
            elif not hand_tile.get("watered_today"):
                op = ["WATER"]
            else:
                op = move
        else:
            op = move
        hands.append(op)

    if len(obs["farms"][obs["player"]]["hands"]) < 5:
        market.append(["HIRE"])

    if obs["step"] == 0 or crop_seeds <= 5:
        market.append(["BUY_SEED", crop_to_plant, 6 - crop_seeds])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    melon_in_shed = obs["private"]["shed"].get("MELON", 0)
    fertilizer_in_shed = obs["private"]["shed"].get("FERTILIZER", 0)
    milk_in_shed = obs["private"]["shed"].get("MILK", 0)
    if wheat_in_shed > 1:
        market.append(["SELL", "WHEAT", wheat_in_shed - 1])
    if melon_in_shed > 0:
        market.append(["SELL", "MELON", melon_in_shed])
    if fertilizer_in_shed > 0:
        market.append(["SELL", "FERTILIZER", fertilizer_in_shed])
    # TODO: SELL MILK here too, once hand 2 is producing it.
    if milk_in_shed > 0:
        market.append(["SELL", "MILK", milk_in_shed]) 
    if tile is None and crop_seeds > 0:
        farmer = ["PLANT", crop_to_plant]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        threshold = 10 if tile["crop"] == "MELON" else 2
        if crop_age >= threshold:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]

    return {"farmer": farmer, "hands": hands, "market": market}

In [4]:
evaluate(melon_then_wheat_goose_cow_agent, melon_then_wheat_goose_agent, n=10)

{'win_rate': 1.0, 'avg_money': 31831.0}

In [5]:
evaluate(melon_then_wheat_goose_cow_agent, melon_then_wheat_goose_agent, n=10)

{'win_rate': 1.0, 'avg_money': 31813.0}

In [6]:
test_env_melon = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
test_env_melon.run([melon_then_wheat_goose_cow_agent, melon_then_wheat_goose_agent])

prev = None
for i, step in enumerate(test_env_melon.steps):
    obs = step[0]["observation"]
    farm = obs["farms"][obs["player"]]
    hands_pos = farm["hands"]
    if len(hands_pos) <= 2:
        continue

    hx, hy = hands_pos[2]
    hand2_tile = farm["tiles"][hy][hx]
    hand2_inv = obs["private"]["inventories"][3]
    shed_goose = obs["private"]["shed"].get("COW", 0)
    shed_wheat = obs["private"]["shed"].get("WHEAT", 0)
    money = farm["money"]
    hand4_action = step[0]["action"]["hands"][2] if len(step[0]["action"]["hands"]) > 2 else None
    fertilizer = obs["private"]["shed"].get("FERTILIZER", 0)
    shed_milk = obs["private"]["shed"].get("MILK", 0)

    snapshot = (
        (hx, hy),
        hand2_tile,
        hand2_inv.get("COW", 0),
        hand2_inv.get("WHEAT", 0),
        shed_goose,
        shed_wheat,
        money,
        fertilizer,
        shed_milk,
    )
    if snapshot != prev:
        print(
            i,
            "hand4 pos:", (hx, hy),
            "| tile:", hand2_tile,
            "| hand4 inv COW/WHEAT:", hand2_inv.get("COW", 0), hand2_inv.get("WHEAT", 0),
            "| shed COW/WHEAT:", shed_goose, shed_wheat,
            "| money:", money,
            "| hand4 action:", hand4_action,
            "| fertilizer:", fertilizer,
            "| milk", shed_milk,
        )
        prev = snapshot


3 hand4 pos: (4, 4) | tile: None | hand4 inv COW/WHEAT: 0 0 | shed COW/WHEAT: 1 1 | money: 1710.0 | hand4 action: None | fertilizer: 0 | milk 0
4 hand4 pos: (4, 4) | tile: None | hand4 inv COW/WHEAT: 1 0 | shed COW/WHEAT: 0 1 | money: 1707.0 | hand4 action: ['PICKUP', 'COW', 1] | fertilizer: 0 | milk 0
5 hand4 pos: (4, 4) | tile: None | hand4 inv COW/WHEAT: 1 1 | shed COW/WHEAT: 0 0 | money: 1702.0 | hand4 action: ['PICKUP', 'WHEAT', 1] | fertilizer: 0 | milk 0
6 hand4 pos: (4, 4) | tile: {'kind': 'PLANT', 'crop': 'MELON', 'planted_day': 0, 'watered_today': False, 'consecutive_unwatered': 1, 'yield_units': 1, 'max_lifespan_step': 312, 'fertilized_until_day': -1} | hand4 inv COW/WHEAT: 1 1 | shed COW/WHEAT: 0 1 | money: 1595.0 | hand4 action: ['PLANT', 'MELON'] | fertilizer: 0 | milk 0
7 hand4 pos: (4, 4) | tile: {'kind': 'PLANT', 'crop': 'MELON', 'planted_day': 0, 'watered_today': True, 'consecutive_unwatered': 1, 'yield_units': 1, 'max_lifespan_step': 312, 'fertilized_until_day': -1} 